# 12 - Stage 11, part D: the quantile crossing in the report

**Purpose.** The decision D-33 (2026-11-12) sorts the quantiles of the neural models wherever they are read as a
distribution and keeps the point forecast as the model's 0.5 head; it committed the report to state how often the
quantiles cross and what that costs. Part D measures both on the final run, on validation and on test, and writes
them into section 6.5 of the report with the two sources of the method (user request, 2026-11-16).

**Rules.**
- Nothing is trained and the step `test` is not run again; the quantile files are the frozen files of the run,
  checked against their step records.
- Nothing is decided from these numbers: no score, no point forecast and no test changes.
- The sources are verified against Crossref and OpenAlex with the user's approval.

**Documented in** `docs/stage11_plan.md` section 7, OP-69, run journal R11-06, errata X117 and X118, tests T-191 and
T-192, and the figure S11-21.

## 1. How often the quantiles are out of order

In [1]:
import pandas as pd

from solarpv.config.paths import REPO_ROOT

t = pd.read_csv(REPO_ROOT / "outputs/11_analyses/quantile_crossing.csv")
t.groupby(["model", "target", "split"])["crossed_pct"].mean().unstack("split").round(1)

split               test   val
model     target              
lstm_sota direct     0.0   0.0
          indirect   0.0   0.0
tft       direct    26.7  56.1
          indirect   1.9   1.8
tft_168   direct    43.1  53.0
          indirect   0.7   0.7
tft_96    direct    65.2  83.5
          indirect  33.7  62.7

**What it shows.** The share of the scored hours (daytime with a measured output) whose quantiles are out of order,
in percent, per model and route, the mean over the horizons and the seeds.

**What it means.** The quantiles of the TFTs are out of order on 0.7 to 83.5 percent of the hours of a
model and route on validation and on 0.7 to 65.2 percent on test, most often for TFT-96 on the direct
route in both periods. The BiLSTM-Opt writes 3 quantiles and
they are never out of order; the TFTs write 7, from separate
heads that nothing in the training keeps in order.

## 2. The table is on the scored hours

In [2]:
m = pd.read_csv(REPO_ROOT / "outputs/11_analyses/metrics_kw.csv")
keys = ["model", "seed", "target", "horizon", "split"]
n = m[m["view"] == "all"].set_index(keys)["n"]
(t.set_index(keys)["n"] == n.reindex(t.set_index(keys).index)).value_counts()

n
True    72
Name: count, dtype: int64

**What it shows.** For every row of the table, whether its number of hours equals that of the scored metrics in kW.

**What it means.** All 72 rows match: the crossing is measured on exactly the hours on which every model is
scored, not on night hours or hours without a measurement.

## 3. What sorting costs the point forecast

In [3]:
tft = t[t["model"].isin(["tft_96", "tft_168", "tft"])]
tft.groupby(["model", "target", "split"])[["point_not_median_pct", "gap_mean_kw", "gap_max_kw"]].mean().round(1)

point_not_median_pct  gap_mean_kw  gap_max_kw
model   target   split                                               
tft     direct   test                   10.6         13.7        61.4
                 val                    23.1         13.5        62.0
        indirect test                    1.4         10.3        26.7
                 val                     0.3          3.7        11.4
tft_168 direct   test                    9.8         11.3        48.2
                 val                    21.3          9.3        33.4
        indirect test                    0.0          0.0         0.0
                 val                     0.0          1.8         1.8
tft_96  direct   test                   25.0         23.3        91.3
                 val                    44.3         25.6       100.1
        indirect test                   33.3         24.3        58.9
                 val                    61.0         20.2        56.7

**What it shows.** Per TFT and route: the share of the hours on which the 0.5 quantile, the point forecast, is not
the median of the sorted quantiles, and the mean and largest distance between the two on those hours, in kW.

**What it means.** On up to 61.0 percent of the hours of
a model and route on validation and 33.3 percent on test
the point forecast is not the median of the sorted distribution, by 19.2 and 20.7 kW on
average on those hours, and by at most 102.2 kW. This is the cost that D-33 accepted: the point forecast stays
the model's own output, so every score of the report is unchanged.

## 4. Sorting never raises the quantile loss

In [4]:
drop = 100 * (1 - t["pinball_sorted_kw"] / t["pinball_raw_kw"])
t.assign(drop_pct=drop).groupby(["model", "target", "split"])["drop_pct"].max().unstack("split").round(2)

split               test   val
model     target              
lstm_sota direct    0.00  0.00
          indirect  0.00  0.00
tft       direct    0.34  1.46
          indirect  0.01  0.01
tft_168   direct    0.35  0.64
          indirect  0.00  0.00
tft_96    direct    1.81  4.56
          indirect  0.52  1.52

**What it shows.** The largest drop of the mean quantile loss in kW over the horizons and seeds, in percent, when the
quantiles are sorted per hour.

**What it means.** The loss falls in 36 of 72 cells and rises in none, by at most 4.6
percent (TFT-96, direct route, validation). The part of the quantile loss that depends on the order of the quantiles is
smallest when they rise with their levels, so sorting can only lower it; this is the monotone rearrangement of
Chernozhukov, Fernández-Val and Galichon (2010).

## 5. Section 6.5 of the report

In [5]:
report = (REPO_ROOT / "docs/report_v7.md").read_text(encoding="utf-8")
print(report.split("### 6.5 Quantile outputs", 1)[1].split("\n## ", 1)[0].strip())

The BiLSTM-Opt forecasts 3 quantiles of the power and the TFTs
7; the point forecast of each model is its
0.5 quantile, as in v6, and only the point
forecast is scored in this report (D-33). The quantile heads of the TFT are separate outputs of one layer, and nothing in
the training keeps them in order. On the scored hours, averaged over the horizons, the quantiles of the TFTs are out of
order on 0.7 to
83.5 percent of the hours on
validation and on 0.7 to
65.2 percent on test, depending on
the model and route, most often for TFT-96 on the direct route; the quantiles of the BiLSTM-Opt are never out of order.

Wherever the quantiles are read as a distribution, they are first sorted per hour: the monotone rearrangement
[Chernozhukov et al., 2010]. The quantile loss [Koenker & Bassett, 1978] of an hour never rises when its quantiles are sorted, because the
part of the loss that depends on their order is smallest when they rise with their levels; on this run sorting lowers
the mean quantil

**What it shows.** The text of section 6.5 as the report generator wrote it.

**What it means.** Every number in it is registered in `docs/traceability.csv` with this table as its source (T-178,
T-192), and it cites the two sources of the method, Chernozhukov et al. (2010) for the rearrangement and Koenker and
Bassett (1978) for the quantile loss, both listed in chapter 17. The figure S11-21 gives the same numbers: the highest
share of crossed hours on validation is 83.5 percent in both.